In [1]:
import os

import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ["WANDB_API_KEY"] = (
    ""
)



In [2]:
# Cell 1: load runs into a DataFrame
import pandas as pd
import numpy as np
import wandb

api = wandb.Api()

PROJECT = "julian_oelhaf/fc-fl-comparison"
runs = api.runs(PROJECT)

records = []
for run in runs:
    summary = run.summary._json_dict
    config = {k: v for k, v in run.config.items() if not k.startswith("_")}

    record = {
        "run_id": run.id,
        "run_name": run.name,
        "state": run.state,
        "created_at": run.created_at,
        "group": run.group,
        "job_type": run.job_type,
        "tags": ",".join(run.tags) if run.tags else "",
    }

    record.update({f"summary.{k}": v for k, v in summary.items()})
    record.update({f"config.{k}": v for k, v in config.items()})

    records.append(record)

runs_df = pd.DataFrame(records)
runs_df = runs_df[runs_df["state"] == "finished"].copy()

print(f"Loaded {len(runs_df)} finished runs")
print(runs_df.shape)

wandb: Currently logged in as: julian-oelhaf (julian_oelhaf) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loaded 630 finished runs
(630, 126)


In [3]:
runs_df

,run_id,run_name,state,created_at,group,job_type,tags,summary._runtime,summary._step,summary._timestamp,...,summary.runtime/predict_single_mean_s/std,summary.runtime/predict_single_std_s/mean,summary.runtime/predict_single_std_s/std,summary.runtime/predict_throughput_samples_per_s/mean,summary.runtime/predict_throughput_samples_per_s/std,summary.runtime/scaler_fit_transform_train_s/mean,summary.runtime/scaler_fit_transform_train_s/std,summary.runtime/scaler_transform_test_s/mean,summary.runtime/scaler_transform_test_s/std,config.runtime_protocol
0,3kt9ysi5,snowy-bee-134,finished,2026-01-21T09:43:37Z,None,None,,384.0,0.0,1.768989e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9knh61ij,fiery-glade-135,finished,2026-01-21T09:48:44Z,None,None,,339.0,0.0,1.768989e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ibj0j5wf,vague-haze-137,finished,2026-01-21T09:54:50Z,None,None,,397.0,0.0,1.768990e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,grxutsz8,efficient-feather-138,finished,2026-01-21T09:58:59Z,None,None,,712.0,0.0,1.768990e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,vku5ihg1,sleek-microwave-139,finished,2026-01-21T10:01:52Z,None,None,,451.0,0.0,1.768990e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
651,gdkan367,summer-wave-920,finished,2026-04-22T11:47:16Z,None,None,,4989.0,4.0,1.776863e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
652,rt0bayn4,devout-sunset-921,finished,2026-04-22T11:52:40Z,None,None,,6490.0,13.0,1.776865e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
653,813djrmr,breezy-butterfly-922,finished,2026-04-22T12:17:53Z,None,None,,5379.0,4.0,1.776866e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
654,95szre1p,twilight-aardvark-923,finished,2026-04-22T12:22:53Z,None,None,,8584.0,13.0,1.776869e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Get default runs with tag "default" or "baseline"
default_runs = runs_df[
    runs_df["tags"].str.contains("default|baseline", case=False, na=False)
].copy()
print(f"Found {len(default_runs)} default runs")

Found 20 default runs


In [5]:
runs_df = runs_df[runs_df["state"] == "finished"].copy()
print(f"Total finished runs: {len(runs_df)}")

Total finished runs: 630


In [6]:
import pandas as pd
import numpy as np

# ============================================================
# 1) Load and basic cleanup
# ============================================================
df = runs_df.copy()


def parse_tags(x):
    if pd.isna(x):
        return set()
    return {t.strip() for t in str(x).split(",") if t.strip()}


df["tags_set"] = df["tags"].map(parse_tags)
df["is_default_tag"] = df["tags_set"].map(lambda s: "default" in s)

print(f"Total runs: {len(df)}")
print(f"Runs tagged as default: {df['is_default_tag'].sum()}")

Total runs: 630
Runs tagged as default: 20


In [12]:
# keep only completed/usable runs
if "summary.completed" in df.columns:
    df = df[df["summary.completed"] == True].copy()

if "state" in df.columns:
    df = df[
        df["state"]
        .astype(str)
        .str.lower()
        .isin(["finished", "completed", "finished_run"])
    ].copy()


# ------------------------------------------------------------
# Helper: normalize values for safe equality checks
# ------------------------------------------------------------
import ast
import numpy as np
import pandas as pd


def canon(v):
    if v is None:
        return None

    if np.isscalar(v):
        try:
            if pd.isna(v):
                return None
        except Exception:
            pass

        if isinstance(v, str):
            s = v.strip()
            if s == "" or s.lower() in {"nan", "none", "null"}:
                return None
            try:
                parsed = ast.literal_eval(s)
                return canon(parsed)
            except Exception:
                return s

        if isinstance(v, np.generic):
            return v.item()

        return v

    if isinstance(v, np.ndarray):
        return tuple(canon(x) for x in v.tolist())

    if isinstance(v, (list, tuple)):
        return tuple(canon(x) for x in v)

    if isinstance(v, dict):
        return tuple(sorted((k, canon(val)) for k, val in v.items()))

    return str(v)


def same(a, b):
    return canon(a) == canon(b)


def same_with_col(col, a, b):
    ca, cb = canon(a), canon(b)

    if col == "config.ablation/enabled":
        # treat missing as disabled
        ca = False if ca is None else ca
        cb = False if cb is None else cb

    return ca == cb


# all config columns
config_cols = [c for c in df.columns if c.startswith("config.")]

# these keys define which baseline a run should be compared to
GROUP_KEYS = [
    "config.model/name",
    "config.task/type",
    "config.task/target_label",
    "config.window/length_s",
]

# ============================================================
# 2) Build tagged-default baseline table
# ============================================================
baselines = df[df["is_default_tag"]].copy()

baseline_counts = (
    baselines.groupby(GROUP_KEYS, dropna=False).size().reset_index(name="n")
)
bad_groups = baseline_counts[baseline_counts["n"] != 1]

if not bad_groups.empty:
    raise ValueError(
        "Expected exactly one `default`-tagged baseline per "
        "{model, task, target, window} group, but found violations:\n"
        + bad_groups.to_string(index=False)
    )

baseline_lookup = {}
for _, row in baselines.iterrows():
    key = tuple(canon(row[k]) for k in GROUP_KEYS)
    baseline_lookup[key] = row

# ============================================================
# 3) Approved hyperparameter change-sets
# ============================================================
APPROVED_CHANGESETS = {
    "hgb.learning_rate": {
        "config.hgb/learning_rate",
    },
    "hgb.min_samples_leaf": {
        "config.hgb/min_samples_leaf",
    },
    "hgb.l2_regularization": {
        "config.hgb/l2_regularization",
    },
    "hgb.max_iter": {
        "config.hgb/max_iter",
    },
    "hgb.max_depth": {
        "config.hgb/max_depth",
        "config.hgb/is_depth_limited",
    },
    "mlp.alpha": {
        "config.mlp/alpha",
    },
    "mlp.max_iter": {
        "config.mlp/max_iter",
    },
    "mlp.batch_size": {
        "config.mlp/batch_size",
    },
    "mlp.learning_rate_init": {
        "config.mlp/learning_rate_init",
    },
    "mlp.hidden_layer_sizes": {
        "config.mlp/hidden_layer_sizes",
        # "config.mlp/width_sum",
        # "config.mlp/width_max",
        # "config.mlp/num_layers",
    },
}

CHANGESET_TO_NAME = {frozenset(v): k for k, v in APPROVED_CHANGESETS.items()}

# ============================================================
# 4) Compare every non-default run to its tagged baseline
# ============================================================
records_valid = []
records_rejected = []
records_baseline_equivalent = []

MUST_MATCH_COLS = {
    "config.ablation/enabled",
}

IGNORE_DIFF_COLS = {
    "config.ablation/features_per_relay",
    "config.ablation/mode",
    "config.ablation/n_relays",
    "config.ablation/relay_index",
    "config.ablation/relay_indices",
}

for _, run in df[~df["is_default_tag"]].iterrows():
    key = tuple(canon(run[k]) for k in GROUP_KEYS)

    if key not in baseline_lookup:
        continue

    base = baseline_lookup[key]

    # 1) hard reject if any MUST_MATCH column differs
    must_match_failed = [
        c for c in MUST_MATCH_COLS if c in df.columns and not same_with_col(c, run[c], base[c])
    ]

    if must_match_failed:
        records_rejected.append(
            {
                "run_id": run["run_id"],
                "run_name": run.get("run_name", None),
                "baseline_run_id": base["run_id"],
                "model": run["config.model/name"],
                "task": run["config.task/type"],
                "target": run["config.task/target_label"],
                "window_s": run["config.window/length_s"],
                "reject_reason": "must_match_failed",
                "changed_cols": must_match_failed,
            }
        )
        continue

    # 2) compute differing columns, ignoring selected metadata columns
    differing = set()
    for c in config_cols:
        if c in MUST_MATCH_COLS or c in IGNORE_DIFF_COLS:
            continue
        if not same_with_col(c, run[c], base[c]):
            differing.add(c)

    # baseline-equivalent modulo ignored ablation metadata columns
    if len(differing) == 0:
        records_baseline_equivalent.append(
            {
                "run_id": run["run_id"],
                "run_name": run.get("run_name", None),
                "baseline_run_id": base["run_id"],
                "model": run["config.model/name"],
                "task": run["config.task/type"],
                "target": run["config.task/target_label"],
                "window_s": run["config.window/length_s"],
            }
        )
        continue

    diff_key = frozenset(differing)

    if diff_key in CHANGESET_TO_NAME:
        ablation_name = CHANGESET_TO_NAME[diff_key]
        changed_cols = sorted(differing)

        rec = {
            "run_id": run["run_id"],
            "run_name": run.get("run_name", None),
            "baseline_run_id": base["run_id"],
            "ablation_name": ablation_name,
            "changed_cols": changed_cols,
            "model": run["config.model/name"],
            "task": run["config.task/type"],
            "target": run["config.task/target_label"],
            "window_s": run["config.window/length_s"],
            "mean_f1": run.get("summary.cv/mean_f1_score", np.nan),
            "std_f1": run.get("summary.cv/std_f1_score", np.nan),
            "mean_mae": run.get("summary.cv/mean_mae", np.nan),
            "std_mae": run.get("summary.cv/std_mae", np.nan),
        }

        for c in changed_cols:
            rec[c] = run[c]

        records_valid.append(rec)

    else:
        records_rejected.append(
            {
                "run_id": run["run_id"],
                "run_name": run.get("run_name", None),
                "baseline_run_id": base["run_id"],
                "model": run["config.model/name"],
                "task": run["config.task/type"],
                "target": run["config.task/target_label"],
                "window_s": run["config.window/length_s"],
                "reject_reason": "unapproved_changeset",
                "n_changed_cols": len(differing),
                "changed_cols": sorted(differing),
            }
        )

valid_ablation_runs = pd.DataFrame(records_valid)
rejected_runs = pd.DataFrame(records_rejected)
baseline_equivalent_runs = pd.DataFrame(records_baseline_equivalent)

# ============================================================
# 5) Optional: compact summary for sanity checking
# ============================================================
if not valid_ablation_runs.empty:
    ablation_summary = (
        valid_ablation_runs.groupby(
            ["ablation_name", "model", "task", "target", "window_s"], dropna=False
        )
        .size()
        .reset_index(name="n_runs")
        .sort_values(["ablation_name", "model", "task", "window_s"])
    )
else:
    ablation_summary = pd.DataFrame()

print("Tagged default baselines:", len(baselines))
print("Valid hyperparameter ablation runs:", len(valid_ablation_runs))
print("Rejected non-baseline runs:", len(rejected_runs))
print("Baseline-equivalent runs (modulo ignored cols):", len(baseline_equivalent_runs))

display(ablation_summary)
display(valid_ablation_runs.head(20))
display(rejected_runs.head(20))
display(baseline_equivalent_runs.head(20))

Tagged default baselines: 20
Valid hyperparameter ablation runs: 274
Rejected non-baseline runs: 255
Baseline-equivalent runs (modulo ignored cols): 6


,ablation_name,model,task,target,window_s,n_runs
0,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,0.01,2
1,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,0.02,2
2,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,0.03,2
3,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,0.04,2
4,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,0.05,2
...,...,...,...,...,...,...
93,mlp.max_iter,mlp_regressor,regression,y_fault_location,0.01,3
94,mlp.max_iter,mlp_regressor,regression,y_fault_location,0.02,3
95,mlp.max_iter,mlp_regressor,regression,y_fault_location,0.03,3
96,mlp.max_iter,mlp_regressor,regression,y_fault_location,0.04,3


,run_id,run_name,baseline_run_id,ablation_name,changed_cols,model,task,target,window_s,mean_f1,...,config.hgb/max_depth,config.hgb/learning_rate,config.hgb/min_samples_leaf,config.hgb/l2_regularization,config.hgb/max_iter,config.mlp/alpha,config.mlp/learning_rate_init,config.mlp/batch_size,config.mlp/max_iter,config.mlp/hidden_layer_sizes
0,9knh61ij,fiery-glade-135,v0t86yr5,hgb.max_depth,"[config.hgb/is_depth_limited, config.hgb/max_d...",hist_gradient_boosting_regressor,regression,y_fault_location,0.05,NaN,...,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ibj0j5wf,vague-haze-137,v0t86yr5,hgb.max_depth,"[config.hgb/is_depth_limited, config.hgb/max_d...",hist_gradient_boosting_regressor,regression,y_fault_location,0.05,NaN,...,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,vku5ihg1,sleek-microwave-139,v0t86yr5,hgb.max_depth,"[config.hgb/is_depth_limited, config.hgb/max_d...",hist_gradient_boosting_regressor,regression,y_fault_location,0.05,NaN,...,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0jl20pa7,restful-wildflower-140,v0t86yr5,hgb.learning_rate,[config.hgb/learning_rate],hist_gradient_boosting_regressor,regression,y_fault_location,0.05,NaN,...,NaN,0.03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6o5jgkzy,morning-dawn-142,v0t86yr5,hgb.learning_rate,[config.hgb/learning_rate],hist_gradient_boosting_regressor,regression,y_fault_location,0.05,NaN,...,NaN,0.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ghnurbix,electric-wood-143,v0t86yr5,hgb.min_samples_leaf,[config.hgb/min_samples_leaf],hist_gradient_boosting_regressor,regression,y_fault_location,0.05,NaN,...,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,5mqula8e,worthy-surf-146,v0t86yr5,hgb.l2_regularization,[config.hgb/l2_regularization],hist_gradient_boosting_regressor,regression,y_fault_location,0.05,NaN,...,NaN,NaN,NaN,0.0001,NaN,NaN,NaN,NaN,NaN,NaN
7,hyj9i5w2,scarlet-fire-147,v0t86yr5,hgb.min_samples_leaf,[config.hgb/min_samples_leaf],hist_gradient_boosting_regressor,regression,y_fault_location,0.05,NaN,...,NaN,NaN,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,zv7ydguq,young-fire-148,v0t86yr5,hgb.l2_regularization,[config.hgb/l2_regularization],hist_gradient_boosting_regressor,regression,y_fault_location,0.05,NaN,...,NaN,NaN,NaN,0.0100,NaN,NaN,NaN,NaN,NaN,NaN
9,bapueuys,fearless-monkey-168,kbwet8px,hgb.max_depth,"[config.hgb/is_depth_limited, config.hgb/max_d...",hist_gradient_boosting_regressor,regression,y_fault_location,0.02,NaN,...,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,run_id,run_name,baseline_run_id,model,task,target,window_s,reject_reason,n_changed_cols,changed_cols
0,3kt9ysi5,snowy-bee-134,v0t86yr5,hist_gradient_boosting_regressor,regression,y_fault_location,0.05,unapproved_changeset,2.0,"[config.hgb/is_depth_limited, config.hgb/max_i..."
1,grxutsz8,efficient-feather-138,v0t86yr5,hist_gradient_boosting_regressor,regression,y_fault_location,0.05,unapproved_changeset,2.0,"[config.hgb/is_depth_limited, config.hgb/max_i..."
2,rllw0krj,twilight-sponge-150,7pvhakkn,mlp_regressor,regression,y_fault_location,0.01,unapproved_changeset,4.0,"[config.mlp/hidden_layer_sizes, config.mlp/num..."
3,ml36u6qu,hardy-glitter-151,7pvhakkn,mlp_regressor,regression,y_fault_location,0.01,unapproved_changeset,4.0,"[config.mlp/alpha, config.mlp/num_layers, conf..."
4,3lpg8wxf,ethereal-sun-152,7pvhakkn,mlp_regressor,regression,y_fault_location,0.01,unapproved_changeset,4.0,"[config.mlp/learning_rate_init, config.mlp/num..."
5,m1tyzufm,dauntless-fog-156,7pvhakkn,mlp_regressor,regression,y_fault_location,0.01,unapproved_changeset,4.0,"[config.mlp/hidden_layer_sizes, config.mlp/num..."
6,oo79syi9,silvery-flower-157,7pvhakkn,mlp_regressor,regression,y_fault_location,0.01,unapproved_changeset,4.0,"[config.mlp/alpha, config.mlp/num_layers, conf..."
7,hr03g8ha,glorious-tree-158,7pvhakkn,mlp_regressor,regression,y_fault_location,0.01,unapproved_changeset,4.0,"[config.mlp/learning_rate_init, config.mlp/num..."
8,ov3wo13i,expert-oath-160,7pvhakkn,mlp_regressor,regression,y_fault_location,0.01,unapproved_changeset,4.0,"[config.mlp/max_iter, config.mlp/num_layers, c..."
9,xad3eprm,generous-elevator-162,7pvhakkn,mlp_regressor,regression,y_fault_location,0.01,unapproved_changeset,4.0,"[config.mlp/batch_size, config.mlp/num_layers,..."


,run_id,run_name,baseline_run_id,model,task,target,window_s
0,9bznvvh0,devout-shadow-651,7pvhakkn,mlp_regressor,regression,y_fault_location,0.01
1,55oj7q32,radiant-plant-652,7pvhakkn,mlp_regressor,regression,y_fault_location,0.01
2,vpoimvam,daily-water-653,a54jr3x2,mlp_classifier,multiclass,event_type,0.02
3,pyr3deus,olive-salad-654,px1gnqw9,mlp_regressor,regression,y_fault_location,0.05
4,14amuji4,royal-shadow-655,sgi11ve9,mlp_classifier,multiclass,event_type,0.05
5,ti9q2asr,stoic-sunset-657,a54jr3x2,mlp_classifier,multiclass,event_type,0.02


In [15]:
import pandas as pd
import numpy as np

# ============================================================
# 1) Which config column should be shown as the ablation value?
# ============================================================
ABLATION_VALUE_COL = {
    "hgb.learning_rate": "config.hgb/learning_rate",
    "hgb.min_samples_leaf": "config.hgb/min_samples_leaf",
    "hgb.l2_regularization": "config.hgb/l2_regularization",
    "hgb.max_iter": "config.hgb/max_iter",
    "hgb.max_depth": "config.hgb/max_depth",
    "mlp.alpha": "config.mlp/alpha",
    "mlp.max_iter": "config.mlp/max_iter",
    "mlp.batch_size": "config.mlp/batch_size",
    "mlp.learning_rate_init": "config.mlp/learning_rate_init",
    "mlp.hidden_layer_sizes": "config.mlp/hidden_layer_sizes",
}

# group keys defining one baseline context
GROUP_KEYS = ["model", "task", "target", "window_s"]


# ------------------------------------------------------------
# helper: stable display formatting for ablation values
# ------------------------------------------------------------
def display_value(v):
    vv = canon(v)
    if vv is None:
        return "None"
    if isinstance(vv, tuple):
        return str(tuple(vv))
    return str(vv)


# ============================================================
# 2) Attach a single metric per run, chosen automatically
#    - MAE for regression / localization
#    - macro-F1 for classification
# ============================================================
vr = valid_ablation_runs.copy()

vr["metric_name"] = pd.Series(pd.NA, index=vr.index, dtype="object")
vr.loc[vr["mean_mae"].notna(), "metric_name"] = "MAE"
vr.loc[vr["mean_mae"].isna() & vr["mean_f1"].notna(), "metric_name"] = "macro-F1"

vr["metric_value"] = pd.Series(np.nan, index=vr.index, dtype="float64")
vr.loc[vr["metric_name"] == "MAE", "metric_value"] = vr.loc[
    vr["metric_name"] == "MAE", "mean_mae"
]
vr.loc[vr["metric_name"] == "macro-F1", "metric_value"] = vr.loc[
    vr["metric_name"] == "macro-F1", "mean_f1"
]

vr["metric_std"] = pd.Series(np.nan, index=vr.index, dtype="float64")
vr.loc[vr["metric_name"] == "MAE", "metric_std"] = vr.loc[
    vr["metric_name"] == "MAE", "std_mae"
]
vr.loc[vr["metric_name"] == "macro-F1", "metric_std"] = vr.loc[
    vr["metric_name"] == "macro-F1", "std_f1"
]

# better-is-lower only for MAE
vr["lower_is_better"] = vr["metric_name"].eq("MAE")


# attach ablation value from the right config column
def extract_ablation_value(row):
    col = ABLATION_VALUE_COL.get(row["ablation_name"], None)
    if col is None:
        return np.nan
    return display_value(row.get(col, np.nan))


vr["ablation_value"] = vr.apply(extract_ablation_value, axis=1)

# drop rows where metric or ablation value could not be determined
vr = vr.dropna(subset=["metric_name", "metric_value", "ablation_value"]).copy()

# ============================================================
# 3) Build baseline metric table from tagged defaults only
# ============================================================
b = baselines.copy()

b = b.rename(
    columns={
        "config.model/name": "model",
        "config.task/type": "task",
        "config.task/target_label": "target",
        "config.window/length_s": "window_s",
    }
)

b["metric_name"] = pd.Series(pd.NA, index=b.index, dtype="object")
b.loc[b["summary.cv/mean_mae"].notna(), "metric_name"] = "MAE"
b.loc[
    b["summary.cv/mean_mae"].isna() & b["summary.cv/mean_f1_score"].notna(),
    "metric_name",
] = "macro-F1"

b["baseline_metric"] = pd.Series(np.nan, index=b.index, dtype="float64")
b.loc[b["metric_name"] == "MAE", "baseline_metric"] = b.loc[
    b["metric_name"] == "MAE", "summary.cv/mean_mae"
]
b.loc[b["metric_name"] == "macro-F1", "baseline_metric"] = b.loc[
    b["metric_name"] == "macro-F1", "summary.cv/mean_f1_score"
]

b["baseline_metric_std"] = pd.Series(np.nan, index=b.index, dtype="float64")
b.loc[b["metric_name"] == "MAE", "baseline_metric_std"] = b.loc[
    b["metric_name"] == "MAE", "summary.cv/std_mae"
]
b.loc[b["metric_name"] == "macro-F1", "baseline_metric_std"] = b.loc[
    b["metric_name"] == "macro-F1", "summary.cv/std_f1_score"
]

baseline_metrics = b[
    GROUP_KEYS + ["metric_name", "baseline_metric", "baseline_metric_std"]
].copy()

# merge baseline metrics into valid ablation runs
vr = vr.merge(
    baseline_metrics,
    on=GROUP_KEYS + ["metric_name"],
    how="left",
    validate="many_to_one",
)

# delta vs baseline
# For readability:
#   - MAE: negative delta is improvement
#   - macro-F1: positive delta is improvement
vr["delta_vs_baseline"] = vr["metric_value"] - vr["baseline_metric"]

# normalized "improvement" score so higher is always better
vr["improvement_vs_baseline"] = np.where(
    vr["metric_name"] == "MAE",
    vr["baseline_metric"] - vr["metric_value"],
    vr["metric_value"] - vr["baseline_metric"],
)

# ============================================================
# 4) Coverage tables
# ============================================================
coverage_long = (
    vr.groupby(
        ["ablation_name", "model", "task", "target", "metric_name", "window_s"],
        dropna=False,
    )
    .agg(
        n_runs=("run_id", "nunique"),
        n_values=("ablation_value", "nunique"),
    )
    .reset_index()
    .sort_values(["ablation_name", "model", "task", "window_s"])
)

coverage_matrix = coverage_long.pivot_table(
    index=["ablation_name", "model", "task", "target", "metric_name"],
    columns="window_s",
    values="n_runs",
    aggfunc="first",
    fill_value=0,
).sort_index()

# ============================================================
# 5) Detailed per-value influence table
# ============================================================
per_value_influence = (
    vr.groupby(
        [
            "ablation_name",
            "model",
            "task",
            "target",
            "metric_name",
            "window_s",
            "ablation_value",
        ],
        dropna=False,
    )
    .agg(
        n_runs=("run_id", "nunique"),
        metric_mean=("metric_value", "mean"),
        metric_std=("metric_value", lambda s: s.std(ddof=1) if len(s) > 1 else 0.0),
        baseline_metric=("baseline_metric", "first"),
        baseline_metric_std=("baseline_metric_std", "first"),
        mean_improvement_vs_baseline=("improvement_vs_baseline", "mean"),
        mean_delta_vs_baseline=("delta_vs_baseline", "mean"),
    )
    .reset_index()
)

# optional rounding for readability
for col in [
    "metric_mean",
    "metric_std",
    "baseline_metric",
    "baseline_metric_std",
    "mean_improvement_vs_baseline",
    "mean_delta_vs_baseline",
]:
    per_value_influence[col] = per_value_influence[col].astype(float)

# ============================================================
# 6) Final influence summary:
#    best / worst / range for each ablation context
# ============================================================
summary_rows = []

for keys, g in per_value_influence.groupby(
    ["ablation_name", "model", "task", "target", "metric_name", "window_s"],
    dropna=False,
):
    g = g.copy()

    metric_name = g["metric_name"].iloc[0]
    lower_is_better = metric_name == "MAE"

    if lower_is_better:
        best_idx = g["metric_mean"].idxmin()
        worst_idx = g["metric_mean"].idxmax()
    else:
        best_idx = g["metric_mean"].idxmax()
        worst_idx = g["metric_mean"].idxmin()

    best = g.loc[best_idx]
    worst = g.loc[worst_idx]

    summary_rows.append(
        {
            "ablation_name": best["ablation_name"],
            "model": best["model"],
            "task": best["task"],
            "target": best["target"],
            "metric_name": best["metric_name"],
            "window_s": best["window_s"],
            "n_runs_total": int(g["n_runs"].sum()),
            "n_values": int(g["ablation_value"].nunique()),
            "baseline_metric": float(best["baseline_metric"]),
            "best_value": best["ablation_value"],
            "best_metric": float(best["metric_mean"]),
            "best_improvement_vs_baseline": float(best["mean_improvement_vs_baseline"]),
            "worst_value": worst["ablation_value"],
            "worst_metric": float(worst["metric_mean"]),
            "effect_range": float(
                worst["metric_mean"] - best["metric_mean"]
                if lower_is_better
                else best["metric_mean"] - worst["metric_mean"]
            ),
        }
    )

influence_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(["ablation_name", "model", "task", "window_s"])
    .reset_index(drop=True)
)

# ============================================================
# 7) Nice presentation columns
# ============================================================
per_value_influence["metric_display"] = (
    per_value_influence["metric_mean"].round(3).astype(str)
    + " ± "
    + per_value_influence["metric_std"].round(3).astype(str)
)

per_value_influence["baseline_display"] = (
    per_value_influence["baseline_metric"].round(3).astype(str)
    + " ± "
    + per_value_influence["baseline_metric_std"].round(3).astype(str)
)

# ============================================================
# 8) Show results
# ============================================================
print("Coverage rows:", len(coverage_long))
print("Per-value influence rows:", len(per_value_influence))
print("Influence summary rows:", len(influence_summary))

display(coverage_long)
display(coverage_matrix)
display(per_value_influence)
display(influence_summary)

Coverage rows: 98
Per-value influence rows: 238
Influence summary rows: 98


,ablation_name,model,task,target,metric_name,window_s,n_runs,n_values
0,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,macro-F1,0.01,2,2
1,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,macro-F1,0.02,2,2
2,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,macro-F1,0.03,2,2
3,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,macro-F1,0.04,2,2
4,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,macro-F1,0.05,2,2
...,...,...,...,...,...,...,...,...
93,mlp.max_iter,mlp_regressor,regression,y_fault_location,MAE,0.01,3,3
94,mlp.max_iter,mlp_regressor,regression,y_fault_location,MAE,0.02,3,3
95,mlp.max_iter,mlp_regressor,regression,y_fault_location,MAE,0.03,3,3
96,mlp.max_iter,mlp_regressor,regression,y_fault_location,MAE,0.04,3,3


window_s                                                                                          0.01  \
ablation_name          model                             task       target           metric_name         
hgb.l2_regularization  hist_gradient_boosting_classifier multiclass event_type       macro-F1        2   
                       hist_gradient_boosting_regressor  regression y_fault_location MAE             4   
hgb.learning_rate      hist_gradient_boosting_classifier multiclass event_type       macro-F1        2   
                       hist_gradient_boosting_regressor  regression y_fault_location MAE             4   
hgb.max_depth          hist_gradient_boosting_classifier multiclass event_type       macro-F1        3   
                       hist_gradient_boosting_regressor  regression y_fault_location MAE             6   
hgb.max_iter           hist_gradient_boosting_classifier multiclass event_type       macro-F1        2   
                       hist_gradient_boosting_regressor  regression y_fault_location MAE             4   
hgb.min_samples_leaf   hist_gradient_boosting_classifier multiclass event_type       macro-F1        2   
                       hist_gradient_boosting_regressor  regression y_fault_location MAE             4   
mlp.alpha              mlp_classifier                    multiclass event_type       macro-F1        4   
                       mlp_regressor                     regression y_fault_location MAE             4   
mlp.batch_size         mlp_classifier                    multiclass event_type       macro-F1        3   
                       mlp_regressor                     regression y_fault_location MAE             3   
mlp.hidden_layer_sizes mlp_classifier                    multiclass event_type       macro-F1        6   
                       mlp_regressor                     regression y_fault_location MAE             3   
mlp.learning_rate_init mlp_classifier                    multiclass event_type       macro-F1        3   
                       mlp_regressor                     regression y_fault_location MAE             3   
mlp.max_iter           mlp_classifier                    multiclass event_type       macro-F1        3   
                       mlp_regressor                     regression y_fault_location MAE             3   

window_s                                                                                          0.02  \
ablation_name          model                             task       target           metric_name         
hgb.l2_regularization  hist_gradient_boosting_classifier multiclass event_type       macro-F1        2   
                       hist_gradient_boosting_regressor  regression y_fault_location MAE             2   
hgb.learning_rate      hist_gradient_boosting_classifier multiclass event_type       macro-F1        2   
                       hist_gradient_boosting_regressor  regression y_fault_location MAE             2   
hgb.max_depth          hist_gradient_boosting_classifier multiclass event_type       macro-F1        3   
                       hist_gradient_boosting_regressor  regression y_fault_location MAE             3   
hgb.max_iter           hist_gradient_boosting_classifier multiclass event_type       macro-F1        2   
                       hist_gradient_boosting_regressor  regression y_fault_location MAE             2   
hgb.min_samples_leaf   hist_gradient_boosting_classifier multiclass event_type       macro-F1        2   
                       hist_gradient_boosting_regressor  regression y_fault_location MAE             2   
mlp.alpha              mlp_classifier                    multiclass event_type       macro-F1        2   
                       mlp_regressor                     regression y_fault_location MAE             2   
mlp.batch_size         mlp_classifier                    multiclass event_type       macro-F1        3   
                       mlp_regressor                     regression y_fault_l

,ablation_name,model,task,target,metric_name,window_s,ablation_value,n_runs,metric_mean,metric_std,baseline_metric,baseline_metric_std,mean_improvement_vs_baseline,mean_delta_vs_baseline,metric_display,baseline_display
0,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,macro-F1,0.01,0.0001,1,0.418005,0.0,0.418111,0.008956,-0.000106,-0.000106,0.418 ± 0.0,0.418 ± 0.009
1,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,macro-F1,0.01,0.01,1,0.773242,0.0,0.418111,0.008956,0.355131,0.355131,0.773 ± 0.0,0.418 ± 0.009
2,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,macro-F1,0.02,0.0001,1,0.737630,0.0,0.738005,0.022940,-0.000376,-0.000376,0.738 ± 0.0,0.738 ± 0.023
3,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,macro-F1,0.02,0.01,1,0.953724,0.0,0.738005,0.022940,0.215719,0.215719,0.954 ± 0.0,0.738 ± 0.023
4,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,macro-F1,0.03,0.0001,1,0.977156,0.0,0.977040,0.001533,0.000116,0.000116,0.977 ± 0.0,0.977 ± 0.002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233,mlp.max_iter,mlp_regressor,regression,y_fault_location,MAE,0.04,300.0,1,10.457177,0.0,10.457177,0.521220,0.000000,0.000000,10.457 ± 0.0,10.457 ± 0.521
234,mlp.max_iter,mlp_regressor,regression,y_fault_location,MAE,0.04,400.0,1,10.457177,0.0,10.457177,0.521220,0.000000,0.000000,10.457 ± 0.0,10.457 ± 0.521
235,mlp.max_iter,mlp_regressor,regression,y_fault_location,MAE,0.05,100.0,1,10.288110,0.0,9.914551,0.331382,-0.373559,0.373559,10.288 ± 0.0,9.915 ± 0.331
236,mlp.max_iter,mlp_regressor,regression,y_fault_location,MAE,0.05,300.0,1,9.914551,0.0,9.914551,0.331382,0.000000,0.000000,9.915 ± 0.0,9.915 ± 0.331


,ablation_name,model,task,target,metric_name,window_s,n_runs_total,n_values,baseline_metric,best_value,best_metric,best_improvement_vs_baseline,worst_value,worst_metric,effect_range
0,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,macro-F1,0.01,2,2,0.418111,0.01,0.773242,3.551312e-01,0.0001,0.418005,0.355237
1,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,macro-F1,0.02,2,2,0.738005,0.01,0.953724,2.157190e-01,0.0001,0.737630,0.216095
2,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,macro-F1,0.03,2,2,0.977040,0.01,0.977645,6.047083e-04,0.0001,0.977156,0.000489
3,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,macro-F1,0.04,2,2,0.981919,0.01,0.981920,1.292511e-06,0.0001,0.981593,0.000327
4,hgb.l2_regularization,hist_gradient_boosting_classifier,multiclass,event_type,macro-F1,0.05,2,2,0.981921,0.01,0.982528,6.065863e-04,0.0001,0.982031,0.000497
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,mlp.max_iter,mlp_regressor,regression,y_fault_location,MAE,0.01,3,3,10.661781,300.0,10.572239,8.954195e-02,100.0,10.942091,0.369852
94,mlp.max_iter,mlp_regressor,regression,y_fault_location,MAE,0.02,3,3,10.195141,300.0,10.195141,0.000000e+00,100.0,10.214123,0.018983
95,mlp.max_iter,mlp_regressor,regression,y_fault_location,MAE,0.03,3,3,10.178999,300.0,10.178999,1.078407e-09,100.0,10.418924,0.239925
96,mlp.max_iter,mlp_regressor,regression,y_fault_location,MAE,0.04,3,3,10.457177,300.0,10.457177,0.000000e+00,100.0,10.511295,0.054118


In [14]:
rejected_runs

,run_id,run_name,baseline_run_id,model,task,target,window_s,reject_reason,n_changed_cols,changed_cols
0,3kt9ysi5,snowy-bee-134,v0t86yr5,hist_gradient_boosting_regressor,regression,y_fault_location,0.05,unapproved_changeset,2.0,"[config.hgb/is_depth_limited, config.hgb/max_i..."
1,grxutsz8,efficient-feather-138,v0t86yr5,hist_gradient_boosting_regressor,regression,y_fault_location,0.05,unapproved_changeset,2.0,"[config.hgb/is_depth_limited, config.hgb/max_i..."
2,rllw0krj,twilight-sponge-150,7pvhakkn,mlp_regressor,regression,y_fault_location,0.01,unapproved_changeset,4.0,"[config.mlp/hidden_layer_sizes, config.mlp/num..."
3,ml36u6qu,hardy-glitter-151,7pvhakkn,mlp_regressor,regression,y_fault_location,0.01,unapproved_changeset,4.0,"[config.mlp/alpha, config.mlp/num_layers, conf..."
4,3lpg8wxf,ethereal-sun-152,7pvhakkn,mlp_regressor,regression,y_fault_location,0.01,unapproved_changeset,4.0,"[config.mlp/learning_rate_init, config.mlp/num..."
...,...,...,...,...,...,...,...,...,...,...
250,o0eocju1,pretty-durian-875,sgi11ve9,mlp_classifier,multiclass,event_type,0.05,must_match_failed,NaN,[config.ablation/enabled]
251,rip2511m,fanciful-sky-876,px1gnqw9,mlp_regressor,regression,y_fault_location,0.05,must_match_failed,NaN,[config.ablation/enabled]
252,tpr7bx2u,glowing-deluge-878,px1gnqw9,mlp_regressor,regression,y_fault_location,0.05,must_match_failed,NaN,[config.ablation/enabled]
253,ep4rm3oy,soft-wind-879,px1gnqw9,mlp_regressor,regression,y_fault_location,0.05,must_match_failed,NaN,[config.ablation/enabled]
